# Predictive Modeling

In [31]:
# Importing my new CSV as my model data

import pandas as pd

crfedf = pd.read_csv("../data/processed/crfedf.csv")

In [32]:
crfedf.head()

,player_name,position,class_year,previous_school,school_name,division,conference_code,primary_conference,source_url,collected_at,height_inches,hometown_city,hometown_state,hometown_country,season,height_imputed
0,Becca Siedenburg,S,SR,Gardner-Webb,Abilene Christian University,DI,WAC,Western Athletic Conference,https://acusports.com/sports/womens-volleyball...,2026-07-20T00:46:56.672717+00:00,68.0,Wales,WI,USA,2025,False
1,Avery Thaler,MB,SO,NaN,Abilene Christian University,DI,WAC,Western Athletic Conference,https://acusports.com/sports/womens-volleyball...,2026-07-20T00:46:56.672717+00:00,70.0,Fairfield,TX,USA,2025,False
2,Rachel Koss,S,JR,NaN,Abilene Christian University,DI,WAC,Western Athletic Conference,https://acusports.com/sports/womens-volleyball...,2026-07-20T00:46:56.672717+00:00,71.0,Freedom,WI,USA,2025,False
3,Courtney Church,OH,R-FR,NaN,Abilene Christian University,DI,WAC,Western Athletic Conference,https://acusports.com/sports/womens-volleyball...,2026-07-20T00:46:56.672717+00:00,72.0,Driftwood,TX,USA,2025,False
4,Hannah Gonzalez,MB,JR,NaN,Abilene Christian University,DI,WAC,Western Athletic Conference,https://acusports.com/sports/womens-volleyball...,2026-07-20T00:46:56.672717+00:00,74.0,Lucas,TX,USA,2025,False


## Inserting Packages

In [33]:
import numpy as np
import pandas as pd

In [34]:
# Making the class years a numerical value for totals
class_map = {
    "FR": 1,
    "R-FR": 2,
    "SO": 2,
    "R-SO": 3,
    "JR": 3,
    "R-JR": 4,
    "SR": 4,
    "R-SR": 5,
    "GR": 5,
    "GRAD": 5,
    "GRADUATE": 5
}

crfedf["experience"] = (
    crfedf["class_year"]
    .astype(str)
    .str.upper()
    .str.strip()
    .map(class_map)
)

In [35]:
# Indicating number of transfers
crfedf["is_transfer"] = crfedf["previous_school"].notna()

In [36]:
# Indicating if the player is international
crfedf["is_international"] = (
    crfedf["hometown_country"]
    .fillna("USA")
    .str.upper()
    != "USA"
)

In [37]:
# State Lookup to convert to state codes
state_lookup = {
    "Alabama": "AL",
    "Alaska": "AK",
    "Arizona": "AZ",
    # ...
}

## Creating "teamdf"

In [38]:
teamdf = (
    crfedf
    .groupby(
        ["school_name", "season"],
        as_index=False
    )
)

In [39]:
# Aggregating the teamdf
teamdf = (
    crfedf
    .groupby(
        ["school_name", "season"],
        as_index=False
    )
    .agg(
        roster_size=("player_name", "count"),

        avg_height=("height_inches", "mean"),
        std_height=("height_inches", "std"),

        avg_experience=("experience", "mean"),

        transfers=("is_transfer", "sum"),

        international_players=("is_international", "sum"),

        conference=("conference_code", "first"),

        division=("division", "first")
    )
)

In [40]:
# Review class-year values that were not included in class_map
unmapped_classes = (
    crfedf.loc[crfedf["experience"].isna(), "class_year"]
    .value_counts(dropna=False)
)

unmapped_classes

class_year
NaN    157
Name: count, dtype: int64

In [41]:
crfedf[
    [
        "class_year",
        "experience",
        "previous_school",
        "is_transfer",
        "hometown_country",
        "is_international",
    ]
].head(10)

,class_year,experience,previous_school,is_transfer,hometown_country,is_international
0,SR,4.0,Gardner-Webb,True,USA,False
1,SO,2.0,NaN,False,USA,False
2,JR,3.0,NaN,False,USA,False
3,R-FR,2.0,NaN,False,USA,False
4,JR,3.0,NaN,False,USA,False
5,SO,2.0,NaN,False,USA,False
6,SR,4.0,NaN,False,USA,False
7,JR,3.0,Cal State Fullerton,True,USA,False
8,JR,3.0,NaN,False,USA,False
9,JR,3.0,NaN,False,USA,False


In [42]:
# Aggregate player-level records into school-season features
teamdf = (
    crfedf
    .groupby(
        ["school_name", "season"],
        as_index=False,
        dropna=False,
    )
    .agg(
        division=("division", "first"),
        conference_code=("conference_code", "first"),
        primary_conference=("primary_conference", "first"),

        roster_size=("player_name", "count"),

        avg_height_inches=("height_inches", "mean"),
        median_height_inches=("height_inches", "median"),
        min_height_inches=("height_inches", "min"),
        max_height_inches=("height_inches", "max"),
        height_std_inches=("height_inches", "std"),
        imputed_height_count=("height_imputed", "sum"),

        avg_experience=("experience", "mean"),
        median_experience=("experience", "median"),
        max_experience=("experience", "max"),

        transfer_count=("is_transfer", "sum"),
        international_count=("is_international", "sum"),
    )
)

In [43]:
# Create roster-relative features
teamdf["transfer_pct"] = (
    teamdf["transfer_count"] / teamdf["roster_size"]
)

teamdf["international_pct"] = (
    teamdf["international_count"] / teamdf["roster_size"]
)

teamdf["imputed_height_pct"] = (
    teamdf["imputed_height_count"] / teamdf["roster_size"]
)

In [44]:
teamdf.head()

,school_name,season,division,conference_code,primary_conference,roster_size,avg_height_inches,median_height_inches,min_height_inches,max_height_inches,height_std_inches,imputed_height_count,avg_experience,median_experience,max_experience,transfer_count,international_count,transfer_pct,international_pct,imputed_height_pct
0,Abilene Christian University,2025,DI,WAC,Western Athletic Conference,15,70.666667,71.0,61.0,76.0,3.498299,0,2.600000,3.0,4.0,4,0,0.266667,0.000000,0.0
1,Alabama A&M University,2025,DI,SWAC,Southwestern Athletic Conf.,17,70.823529,72.0,65.0,75.0,2.811479,0,2.470588,2.0,5.0,0,2,0.000000,0.117647,0.0
2,Alabama State University,2025,DI,SWAC,Southwestern Athletic Conf.,13,69.461538,70.0,63.0,74.0,3.152126,0,2.538462,2.0,5.0,0,0,0.000000,0.000000,0.0
3,Alcorn State University,2025,DI,SWAC,Southwestern Athletic Conf.,7,68.857143,70.0,61.0,74.0,4.845223,0,2.142857,2.0,5.0,0,1,0.000000,0.142857,0.0
4,Appalachian State University,2025,DI,SBC,Sun Belt Conference,10,71.900000,71.5,70.0,75.0,1.852926,0,3.000000,3.0,5.0,0,1,0.000000,0.100000,0.0


In [45]:
experience_counts = (
    pd.crosstab(
        index=[
            crfedf["school_name"],
            crfedf["season"],
        ],
        columns=crfedf["experience"],
    )
    .rename(
        columns={
            1.0: "experience_year_1_count",
            2.0: "experience_year_2_count",
            3.0: "experience_year_3_count",
            4.0: "experience_year_4_count",
            5.0: "experience_year_5_count",
        }
    )
    .reset_index()
)

experience_counts.head()

experience,school_name,season,experience_year_1_count,experience_year_2_count,experience_year_3_count,experience_year_4_count,experience_year_5_count
0,Abilene Christian University,2025,2,4,7,2,0
1,Alabama A&M University,2025,6,3,3,4,1
2,Alabama State University,2025,3,4,3,2,1
3,Alcorn State University,2025,3,2,1,0,1
4,Appalachian State University,2025,0,4,3,2,1


In [46]:
experience_columns = [
    "experience_year_1_count",
    "experience_year_2_count",
    "experience_year_3_count",
    "experience_year_4_count",
    "experience_year_5_count",
]

for column in experience_columns:
    if column not in experience_counts.columns:
        experience_counts[column] = 0

In [47]:
teamdf = teamdf.merge(
    experience_counts,
    on=["school_name", "season"],
    how="left",
)

In [48]:
for column in experience_columns:
    pct_column = column.replace("_count", "_pct")

    teamdf[pct_column] = (
        teamdf[column] / teamdf["roster_size"]
    )

In [49]:
crfedf["position"].value_counts(dropna=False)

position
OH      652
MB      548
S       382
L/DS    345
OPP     164
NaN      32
Name: count, dtype: int64

In [50]:
position_counts = (
    pd.crosstab(
        index=[
            crfedf["school_name"],
            crfedf["season"],
        ],
        columns=crfedf["position"],
    )
    .add_prefix("position_")
    .add_suffix("_count")
    .reset_index()
)

position_counts.head()

position,school_name,season,position_L/DS_count,position_MB_count,position_OH_count,position_OPP_count,position_S_count
0,Abilene Christian University,2025,2,3,8,0,2
1,Alabama A&M University,2025,3,6,6,0,2
2,Alabama State University,2025,2,4,4,1,2
3,Alcorn State University,2025,2,2,2,0,1
4,Appalachian State University,2025,0,3,3,1,3


In [51]:
position_counts.columns = (
    position_counts.columns
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace("/", "_", regex=False)
    .str.replace("-", "_", regex=False)
    .str.replace(" ", "_", regex=False)
)

In [52]:
position_counts = position_counts.rename(
    columns={
        "school_name": "school_name",
        "season": "season",
    }
)

In [53]:
teamdf = teamdf.merge(
    position_counts,
    on=["school_name", "season"],
    how="left",
)

In [54]:
print(f"Player-level shape: {crfedf.shape}")
print(f"Team-season shape: {teamdf.shape}")
print(
    "Unique school-seasons:",
    crfedf[["school_name", "season"]]
    .drop_duplicates()
    .shape[0],
)

Player-level shape: (2123, 19)
Team-season shape: (157, 35)
Unique school-seasons: 157


In [55]:
teamdf.info()

<class 'pandas.DataFrame'>
RangeIndex: 157 entries, 0 to 156
Data columns (total 35 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   school_name              157 non-null    str    
 1   season                   157 non-null    int64  
 2   division                 157 non-null    str    
 3   conference_code          157 non-null    str    
 4   primary_conference       157 non-null    str    
 5   roster_size              157 non-null    int64  
 6   avg_height_inches        157 non-null    float64
 7   median_height_inches     157 non-null    float64
 8   min_height_inches        157 non-null    float64
 9   max_height_inches        157 non-null    float64
 10  height_std_inches        155 non-null    float64
 11  imputed_height_count     157 non-null    int64  
 12  avg_experience           148 non-null    float64
 13  median_experience        148 non-null    float64
 14  max_experience           148 non-null

In [56]:
teamdf.head()

,school_name,season,division,conference_code,primary_conference,roster_size,avg_height_inches,median_height_inches,min_height_inches,max_height_inches,...,experience_year_1_pct,experience_year_2_pct,experience_year_3_pct,experience_year_4_pct,experience_year_5_pct,position_l_ds_count,position_mb_count,position_oh_count,position_opp_count,position_s_count
0,Abilene Christian University,2025,DI,WAC,Western Athletic Conference,15,70.666667,71.0,61.0,76.0,...,0.133333,0.266667,0.466667,0.133333,0.000000,2.0,3.0,8.0,0.0,2.0
1,Alabama A&M University,2025,DI,SWAC,Southwestern Athletic Conf.,17,70.823529,72.0,65.0,75.0,...,0.352941,0.176471,0.176471,0.235294,0.058824,3.0,6.0,6.0,0.0,2.0
2,Alabama State University,2025,DI,SWAC,Southwestern Athletic Conf.,13,69.461538,70.0,63.0,74.0,...,0.230769,0.307692,0.230769,0.153846,0.076923,2.0,4.0,4.0,1.0,2.0
3,Alcorn State University,2025,DI,SWAC,Southwestern Athletic Conf.,7,68.857143,70.0,61.0,74.0,...,0.428571,0.285714,0.142857,0.000000,0.142857,2.0,2.0,2.0,0.0,1.0
4,Appalachian State University,2025,DI,SBC,Sun Belt Conference,10,71.900000,71.5,70.0,75.0,...,0.000000,0.400000,0.300000,0.200000,0.100000,0.0,3.0,3.0,1.0,3.0


In [57]:
teamdf.isna().sum().sort_values(ascending=False).head(15)

avg_experience             9
max_experience             9
median_experience          9
experience_year_3_pct      9
experience_year_2_pct      9
experience_year_5_count    9
experience_year_1_pct      9
experience_year_4_count    9
experience_year_3_count    9
experience_year_4_pct      9
experience_year_5_pct      9
experience_year_2_count    9
experience_year_1_count    9
height_std_inches          2
position_s_count           1
dtype: int64

In [58]:
teamdf.to_csv(
    "../data/processed/teamdf.csv",
    index=False,
)